# Delay Prediction using Multi-Output Linear Regression

This notebook covers training multiple Linear Regression models to predict actual logistics delays (Preparation, Internal, and Carrier Transport) based on standard product-level configure targets in the database, trained on **DataCo Supply Chain** dataset.

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymongo import MongoClient
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 1: Connect to Data Source (DataCo CSV or MongoDB)

In [ ]:
import os
import pandas as pd
import numpy as np
from pymongo import MongoClient

loaded_data = False
X_data = []
y_prep, y_internal, y_transport = [], [], []

print("Attempting to load actual dataset from MongoDB...")
try:
    client = MongoClient('mongodb://localhost:27017', serverSelectionTimeoutMS=2000)
    db = client['smart_supply_chain']
    products = list(db["products"].find())
    orders = list(db["sales_orders"].find())
    client.close()
    
    if products and orders:
        products_map = {int(p["sku"]): p for p in products}
        for o in orders:
            lines = o.get("order_lines", [])
            if not lines:
                continue
            preps, internals, transports = [], [], []
            for line in lines:
                sku = int(line.get("product_sku"))
                prod = products_map.get(sku)
                if prod:
                    # mongodb may have missing fields, so we use get with defaults
                    p_val = prod.get("prep_delay")
                    i_val = prod.get("internal_delay")
                    t_val = prod.get("transport_delay")
                    preps.append(4 if p_val is None else p_val)
                    internals.append(0 if i_val is None else i_val)
                    transports.append(0 if t_val is None else t_val)
            if not preps:
                continue
            X_data.append([max(preps), max(internals), max(transports)])
            y_prep.append(o.get("scheduled_shipment", 4))
            y_internal.append(o.get("internalDelay", 0))
            y_transport.append(o.get("transportDelay", 0))
            
        X_arr = np.array(X_data)
        y_prep = np.array(y_prep)
        y_internal = np.array(y_internal)
        y_transport = np.array(y_transport)
        
        if len(X_arr) > 0:
            print(f"Successfully loaded {len(X_arr)} records from MongoDB.")
            loaded_data = True
except Exception as ex:
    print(f"Could not load from MongoDB: {ex}")

   

df_features = pd.DataFrame(X_arr, columns=['Config Prep Delay', 'Config Internal Delay', 'Config Transport Delay'])
df_targets = pd.DataFrame({
    'Actual Scheduled Shipment': y_prep,
    'Actual Internal Delay': y_internal,
    'Actual Transport Delay': y_transport
})
print(f"Loaded {len(df_features)} samples for delay regression.")
df_features.head()

Attempting to load actual dataset from MongoDB...
Successfully loaded 500 records from MongoDB.
Loaded 500 samples for delay regression.


,Config Prep Delay,Config Internal Delay,Config Transport Delay
0,1,4,3
1,5,0,5
2,4,5,5
3,4,5,5
4,5,5,4


### Step 2: Fit Linear Regression Models
We fit three separate regression models, one for each delay output target.

In [8]:
model_prep = LinearRegression().fit(X_arr, y_prep)
model_internal = LinearRegression().fit(X_arr, y_internal)
model_transport = LinearRegression().fit(X_arr, y_transport)

print("1. Preparation Delay Model Coefficients:")
print(f"   Coefficients: {model_prep.coef_}")
print(f"   Intercept:    {model_prep.intercept_:.4f}\n")

print("2. Internal Delay Model Coefficients:")
print(f"   Coefficients: {model_internal.coef_}")
print(f"   Intercept:    {model_internal.intercept_:.4f}\n")

print("3. Transport Delay Model Coefficients:")
print(f"   Coefficients: {model_transport.coef_}")
print(f"   Intercept:    {model_transport.intercept_:.4f}")

1. Preparation Delay Model Coefficients:
   Coefficients: [ 0.11061134 -0.01934803  0.00073699]
   Intercept:    2.9406

2. Internal Delay Model Coefficients:
   Coefficients: [-0.04310003  0.00883948 -0.00520151]
   Intercept:    0.5604

3. Transport Delay Model Coefficients:
   Coefficients: [-0.06421188  0.0059091   0.00758727]
   Intercept:    0.8941


### Step 3: Evaluate Regression Metrics
We compute $R^2$ score (explained variance), Mean Absolute Error (MAE), and Mean Squared Error (MSE) for each target.

In [12]:
pred_prep = model_prep.predict(X_arr)
pred_internal = model_internal.predict(X_arr)
pred_transport = model_transport.predict(X_arr)

targets_pred = [pred_prep, pred_internal, pred_transport]
targets_actual = [y_prep, y_internal, y_transport]
names = ['Preparation Delay', 'Internal Delay', 'Transport Delay']

for i in range(3):

    mae = mean_absolute_error(targets_actual[i], targets_pred[i])
    mse = mean_squared_error(targets_actual[i], targets_pred[i])
    print(f"{names[i]} Evaluation:")
    print(f"  Mean Absolute Error:   {mae:.4f} days")
    print(f"  Mean Squared Error:    {mse:.4f} days^2\n")

Preparation Delay Evaluation:
  Mean Absolute Error:   0.8829 days
  Mean Squared Error:    0.9312 days^2

Internal Delay Evaluation:
  Mean Absolute Error:   0.5325 days
  Mean Squared Error:    0.3947 days^2

Transport Delay Evaluation:
  Mean Absolute Error:   0.6273 days
  Mean Squared Error:    0.5066 days^2

